# Hole Depth Estimation

## The Stone-Drop Method

You can estimate the depth of a hole by dropping a stone and timing how long it takes until you hear it hit the bottom. The total elapsed time includes two components:

1. **Free fall** — the stone accelerates downward under gravity
2. **Sound return** — the impact sound travels back up to your ears at the speed of sound

## Constants

- **Acceleration due to gravity:** $g = 9.8$ m/s²
- **Speed of sound in air:** $v = 343$ m/s
- **Measured total time:** $t_{\text{total}} = x$ seconds

## Forward Model (depth → time)

$$t_{\text{fall}} = \sqrt{\frac{2h}{g}}, \quad t_{\text{sound}} = \frac{h}{v}, \quad t_{\text{total}} = \sqrt{\frac{2h}{g}} + \frac{h}{v}$$

## Inverse Model (time → depth)

Substituting $u = \sqrt{h}$ gives a quadratic in $u$:

$$\frac{u^2}{v} + u\sqrt{\frac{2}{g}} - t_{\text{total}} = 0$$

Taking the positive root:

$$h = \left(\frac{-\sqrt{2/g} + \sqrt{2/g + 4t_{\text{total}}/v}}{2/v}\right)^2$$

In [ ]:
import math

G = 9.8  # m/s^2
SPEED_OF_SOUND = 343  # m/s


def total_time(depth_m: float) -> float:
    """Calculate total time from drop to hearing the impact."""
    fall_time = math.sqrt(2 * depth_m / G)
    sound_time = depth_m / SPEED_OF_SOUND
    return fall_time + sound_time


def estimate_depth(t_total_s: float) -> float:
    """Estimate hole depth from measured total time."""
    if t_total_s <= 0:
        raise ValueError("Total time must be positive")

    sqrt_2_over_g = math.sqrt(2 / G)
    discriminant = 2 / G + 4 * t_total_s / SPEED_OF_SOUND
    u = (-sqrt_2_over_g + math.sqrt(discriminant)) / (2 / SPEED_OF_SOUND)
    return u ** 2

## Why Sound Delay Matters

For shallow holes, the sound return time is a small fraction of the total — most of the delay is free fall. As depth increases, the sound travel time grows linearly while fall time grows with the square root of depth, so the sound component becomes more significant at greater depths.

- [**Speed of sound**](https://en.wikipedia.org/wiki/Speed_of_sound): In dry air at 20 °C, sound travels at approximately 343 m/s

## Create and Display the Plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Set up plotting style (try seaborn, fallback to default)
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    try:
        plt.style.use('seaborn-darkgrid')
    except OSError:
        plt.style.use('default')
        print("Using default matplotlib style")

# Create the plot
fig, ax = plt.subplots(figsize=(12, 8))

times_s = np.linspace(0.5, 10, 200)
depths_m = [estimate_depth(t) for t in times_s]
ax.plot(times_s, depths_m, linewidth=2.5, color='#007AFF', label='Estimated Depth')

# Annotate example times
example_times = {
    '1 s': 1,
    '2 s': 2,
    '3 s': 3,
    '5 s': 5,
}

for label, t in example_times.items():
    depth = estimate_depth(t)
    ax.plot(t, depth, 'o', markersize=12, zorder=5, color='#FF6B6B')
    ax.annotate(
        f'{label} → {depth:.0f} m',
        xy=(t, depth),
        xytext=(15, 15),
        textcoords='offset points',
        bbox=dict(boxstyle='round,pad=0.6', facecolor='white', edgecolor='gray', alpha=0.9, linewidth=1.5),
        fontsize=9,
        ha='left',
    )

ax.grid(True, alpha=0.2)
ax.legend(loc='upper left', fontsize=10, framealpha=0.9)

ax.set_xlabel('Total Time (s)', fontsize=12, fontweight='bold')
ax.set_ylabel('Estimated Depth (m)', fontsize=12, fontweight='bold')
ax.set_title(
    'Hole Depth Estimation from Stone-Drop Timing\n'
    'Accounting for Free Fall and Speed of Sound',
    fontsize=14,
    fontweight='bold',
    pad=20,
)

plt.tight_layout()

output_path = Path('hole_depth_estimation.png')
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f'Plot saved to: {output_path.absolute()}')

plt.show()

# Time breakdown for an example measurement
example_t = 3.0
example_h = estimate_depth(example_t)
fall_t = math.sqrt(2 * example_h / G)
sound_t = example_h / SPEED_OF_SOUND
print(f'\nBreakdown for t_total = {example_t} s:')
print(f'  Estimated depth:  {example_h:.1f} m')
print(f'  Fall time:        {fall_t:.3f} s ({fall_t / example_t * 100:.1f}%)')
print(f'  Sound return:     {sound_t:.3f} s ({sound_t / example_t * 100:.1f}%)')